In [2]:
import cv2
import os
import glob
import numpy as np
from tqdm import tqdm # Thư viện tạo thanh tiến trình (progress bar) rất chuyên nghiệp

# 1. Cấu hình đường dẫn
INPUT_DIR = "../data/raw"
OUTPUT_DIR = "../data/processed"
TARGET_SIZE = (256, 256)

# Đảm bảo thư mục đầu ra tồn tại, nếu chưa có thì tự động tạo
os.makedirs(OUTPUT_DIR, exist_ok=True)

def remove_background_otsu(image):
    """
    Sử dụng Otsu Thresholding để tách lá cây khỏi nền.
    """
    # Chuyển sang ảnh xám
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    
    # Làm mịn ảnh để giảm nhiễu (khử các chi tiết nhỏ li ti trên nền)
    blurred = cv2.GaussianBlur(gray, (7, 7), 0)
    
    # Áp dụng Otsu threshold tự động tìm ngưỡng
    # Tùy thuộc vào nền sáng hay tối mà dùng THRESH_BINARY hay THRESH_BINARY_INV
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU)
    
    # Dùng thuật toán Morphology để lấp đầy các lỗ hổng bên trong lá (nếu có)
    kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
    thresh = cv2.morphologyEx(thresh, cv2.MORPH_CLOSE, kernel)
    
    # Tạo mask nền đen, lá giữ nguyên màu sắc gốc
    result = cv2.bitwise_and(image, image, mask=thresh)
    return result

def resize_and_pad(image, target_size):
    """
    Resize ảnh giữ nguyên tỉ lệ (Aspect Ratio) và đệm viền đen (Zero-padding).
    """
    old_size = image.shape[:2] # (height, width)
    ratio = float(target_size[0]) / max(old_size)
    new_size = tuple([int(x * ratio) for x in old_size])
    
    # Resize
    image_resized = cv2.resize(image, (new_size[1], new_size[0]))
    
    # Tính toán phần viền cần đệm thêm để đủ kích thước target_size
    delta_w = target_size[1] - new_size[1]
    delta_h = target_size[0] - new_size[0]
    top, bottom = delta_h // 2, delta_h - (delta_h // 2)
    left, right = delta_w // 2, delta_w - (delta_w // 2)
    
    # Đệm viền màu đen (0, 0, 0)
    color = [0, 0, 0]
    new_image = cv2.copyMakeBorder(image_resized, top, bottom, left, right, cv2.BORDER_CONSTANT, value=color)
    
    return new_image

def process_all_images():
    """
    Hàm lặp qua tất cả ảnh trong thư mục raw, xử lý và lưu vào thư mục processed.
    """
    # Lấy danh sách tất cả các file có đuôi .jpg, .jpeg, .png (không phân biệt hoa thường)
    search_path = os.path.join(INPUT_DIR, "*.*")
    all_files = glob.glob(search_path)
    
    # Lọc ra chỉ các file hình ảnh
    valid_extensions = ('.jpg', '.jpeg', '.png')
    image_paths = [f for f in all_files if f.lower().endswith(valid_extensions)]
    
    print(f"Tìm thấy {len(image_paths)} ảnh trong thư mục {INPUT_DIR}.")
    print("Bắt đầu tiến trình xử lý hàng loạt...")
    
    # Dùng tqdm để hiển thị thanh tiến trình
    for img_path in tqdm(image_paths, desc="Processing Images"):
        # Đọc ảnh
        image = cv2.imread(img_path)
        
        # Bỏ qua nếu lỗi đọc file
        if image is None:
            print(f"\n[Lỗi] Không thể đọc ảnh: {img_path}")
            continue
            
        try:
            # 1. Tách nền
            img_no_bg = remove_background_otsu(image)
            
            # 2. Chuẩn hóa kích thước
            img_final = resize_and_pad(img_no_bg, TARGET_SIZE)
            
            # 3. Lấy tên file gốc và tạo đường dẫn lưu mới
            filename = os.path.basename(img_path)
            save_path = os.path.join(OUTPUT_DIR, filename)
            
            # Lưu ảnh
            cv2.imwrite(save_path, img_final)
            
        except Exception as e:
            print(f"\n[Lỗi] Quá trình xử lý thất bại tại ảnh {filename}: {str(e)}")
            
    print(f"\nHoàn tất! Ảnh đã được lưu tại thư mục: {OUTPUT_DIR}")

# Chạy script
if __name__ == "__main__":
    process_all_images()

Tìm thấy 1907 ảnh trong thư mục ../data/raw.
Bắt đầu tiến trình xử lý hàng loạt...


Processing Images:   0%|          | 0/1907 [00:00<?, ?it/s]

Processing Images: 100%|██████████| 1907/1907 [01:39<00:00, 19.12it/s]


Hoàn tất! Ảnh đã được lưu tại thư mục: ../data/processed
